# 56. Carrier heads: localisation and overlap on the 3B (exploration, nothing claimed)

**Numbering.** The carrier-heads plan calls this notebook 54. That number went to the risk-control feasibility work (notebooks 54, 54b and 54c), so this is notebook 56. The plan's notebook 55 becomes 57 and its notebook 56 becomes 58.

**Where this starts.** Notebook 53 found that writing the five template-token states of a successful attack into a run on a clean document, with no injection anywhere in the input, makes the model carry out that attack's instruction in 56.5 percent of 147 runs (54.2 percent on the 72 twins of robust successes), against 0.7 percent for another clean document's state and 0 percent for same-size noise built as notebook 53b rebuilt it. So those five positions carry the injected task itself.

**The question.** Which attention heads carry an injected instruction from the document into the template tokens; are they the same heads that carry a legitimate instruction given in the user turn; and is the set small enough to defend at. The first question is a localisation question, the second is the one no paper in the novelty check asks for prompt injection.

**What is patched.** For query head h at layer l, its output is the slice of the input to that layer's attention output projection belonging to h. A patch replaces that slice during the prompt pass, at the five template positions only, from a cached source run. Nothing at the document tokens is touched and decode steps are untouched. Qwen2.5-3B-Instruct has 36 layers and 16 query heads, so 576 heads. The hooks go in a new module, `src/head_patching.py`; `src/task_state.py` and `src/state_patching.py` are not changed.

**Two directions.**
- *Sufficiency*, donor to clean: a head's output from the attacked run is written into the clean twin's run. Does the twin now follow the injection?
- *Necessity*, clean to attacked: a head's output from the clean twin is written into the attacked run. Does the attack stop?

**The cheap effect measure, with two gates.** Generating and judging an answer for every head and item is not affordable, so the sweep scores a patched run by the **injection margin**: the mean log-probability of the first 16 tokens of the attacked run's answer under the patched prompt, minus the same for the clean twin's answer, the two scored as two rows of one batch with the 16 answer tokens appended, so the write lands 16 places before the end. Gate one, before the sweep: the margin must separate runs the judge called followed from the rest, with AUROC at least 0.80, on the full-state version of the effect the sweep measures. Gate two, after the sweep: the margin of the joint C_20 patch must do the same against the judge on the 72 set-check items with AUROC at least 0.75. If gate one fails the sweep does not run. If gate two fails, the top 64 heads are re-ranked by generation and the judge on 24 items.

**Head sets.** Per-head effects are averaged over items and ranked. The carrier set C_k is the top k by sufficiency effect, for k of 10, 20 and 40. The sweep runs on 32 of the 72 robust items drawn by a fixed seed in proportion to family; every set check runs on all 72, so a set is never scored on the data that chose it. Control sets of the same size: random heads matched by layer (three draws), the lowest-effect heads within C_k's own layers, and an attention-mass set in the manner of Attention Tracker.

**Marks, fixed in the first code cell before any result.**

| question | mark |
|---|---|
| Proxy valid (gate one) | AUROC of the injection margin against the judge at least 0.80 on the same-document full-state write (the attacked run's whole template state into its own clean twin). Otherwise the sweep is replaced by generation on 24 items and this notebook stops to report that. Two further versions are reported with no mark: the full-state necessity write from notebook 53's cache, and the cross-document transplant, which was the gate as the plan wrote it. |
| L1: a small set is sufficient | C_20 written from the attacked run into the clean twin makes the twin follow the injection in at least 0.30 of the 72 (the full state gives 0.542 on these 72); each of the three control sets at most 0.05. PARTIAL: C_40 reaches 0.15 with controls at most 0.05. |
| L2: the set is necessary | C_20's outputs from the clean twin written into the attacked run stop at least 0.60; random layer-matched heads at most 0.15; paired difference lower end at least 0.25. Mean ablation and zeroing of C_20 are recorded beside it. |
| L3: content survives | Median token F1 of the stopped answers with the clean twin's answer no more than 0.10 below notebook 53's full-state clean arm (0.597); at most 0.05 degenerate. |
| I1: the instruction machinery works | I_20 written from an instruction run into a plain run makes the model do the task in at least 0.30. If not, the overlap test is void and is reported as such. |
| O1: overlap | *Shared*: C_20 and I_20 share at least 8 heads and above the 99.9th percentile of 10,000 layer-matched random draws, Spearman correlation of the two per-head effect profiles at least 0.4, and cross-transfer at least 0.30. *Separate*: at most 2 shared heads and cross-transfer at most 0.10. Anything else is *partly shared*. Reported per family as well. |

**Changes from the plan, made before any result.**
1. The numbering, as above.
2. The Attention Tracker control set. Its rule needs the injected instruction's token span, which is not recorded for rewritten documents, so the control uses attention mass from the last prompt token to the whole document span. It is the closest recoverable form of that rule and is named as such wherever it appears.
3. The instruction sweep runs on 30 of pool one's 150 items, six per family by a fixed seed, because a full sweep there costs five times the main sweep. The instruction set checks and the cross-transfer use all 150 items of pool two.
4. The complement control is the k lowest-effect heads within C_k's own layers, which the plan left unspecified.
5. Gate two reuses the set-check generations rather than a separate run.
6. **Gate one, changed after it failed as written, and the failure reported.** (Version A is the gate; if the same-document write turns out to follow the injection on every one of the 72, there is only one class and no AUROC, and the gate falls to version B, the same answer pair in the necessity direction, which notebook 53 already showed has both classes. Which version served as the gate is printed and written to the marks table.) The plan set gate one on notebook 53's transplant, whose donor is a robust success from another category and another document. The margin asks whether the donor's own answer text became more likely, and for a cross-document donor that text is about a different document, so for base encoding, ciphers and translation the donor's first 16 tokens cannot be likely however well the injection was followed. The run of that gate on this notebook's first pass failed. It was testing whether the measure recognises another document's answer, which is not what either sweep does: in the sufficiency sweep the source run is the attacked run of the same injection, so its answer tokens are the right target. Gate one is therefore the same-document full-state write, which notebook 53 never ran and which this notebook generates and judges on the 72 twins. The cross-document version is still computed and printed in the marks table as a failed pre-registered check, and the necessity version is printed beside it. The threshold, 0.80, is unchanged. Nothing else in the notebook moved, and no mark on a head set was touched.

**Decision, from the plan.** L1 at k of 20 or 40 with L2 and L3: go to notebook 57 with the reading from O1. L1 fails even at k of 40: the carrier is spread too widely for a head-level defense, the route closes at the head level, and the transplant stays an observation. L1 passes and L2 fails: notebook 57 runs with the defense judged on security alone and the necessity claim dropped. I1 fails: the overlap question is not asked and notebook 57 runs the mean-ablation form only.

**Compute.** No training and no attack run. One head-output capture pass per item; two per-head sweeps (576 heads, 32 items, two rows per item); set checks by generation with the paper's judge; one instruction sweep; one attention pass. Everything is cached per stage and a disconnected session resumes.

**Outputs.** `data/head_patching/`, `reports/head_patching/*.csv`, `reports/head_patching/go.txt`, `figures/head_patching/heads.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first): install from the local disk, then mount ===
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes peft langdetect==1.0.9 scikit-learn
!pip uninstall -y -q torchao   # Colab ships an old torchao that the current PEFT rejects; nothing in this project uses it
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports/head_patching', 'figures/head_patching', 'data/head_patching'): os.makedirs(d_, exist_ok=True)
import torch, gc
assert torch.cuda.is_available(), 'this notebook needs a GPU'
print('Session ready. GPU:', torch.cuda.get_device_name(0))

## Stage A. The literature cell: every paper of the plan's Table 1 opened, the unconfirmed ones read or dropped

Nothing is cited from a search snippet. This cell fetches each listing page and prints its title and abstract, so the six papers the plan marked unconfirmed are read here before any of them is cited. If a page cannot be fetched the paper stays unconfirmed and is not cited.

In [ ]:
import json, re, urllib.request, pandas as pd
REP, FIG, OUT = 'reports/head_patching', 'figures/head_patching', 'data/head_patching'
PAPERS = {
    # confirmed in the plan, re-opened here so the record is in this notebook
    '2310.15213': 'Function Vectors in Large Language Models (Todd et al., ICLR 2024)',
    '2505.12075': 'Do different prompting methods yield a common task representation? (Davidson et al. 2025)',
    '2310.15916': 'In-Context Learning Creates Task Vectors (Hendel et al. 2023)',
    '2609.37737': 'Where Do LLMs Decide to Break the Rules? (Wen et al. 2026)',
    '2610.05295': 'Readable Before Actionable (Yu et al. 2026), partly confirmed in the plan',
    '2607.26228': 'V-Steer (Zeng et al., COLM 2026)',
    '2602.20708': 'ICON (Wang et al. 2026), partly confirmed in the plan',
    '2411.00348': 'Attention Tracker (Hung et al., NAACL 2025 Findings)',
    '2502.13946': 'Why Safeguarded Ships Run Aground? (Leong et al., ACL 2025)',
    '2609.36570': 'CounterSteer (Russinovich 2026)',
    '2602.07930': 'Patches of Nonlinearity: Instruction Vectors (Bigoulaeva et al. 2026)',
    # the six the plan marked not confirmed: read here or dropped
    '2605.24659': 'IterInject (not confirmed in the plan)',
    '2608.08027': 'BASIS (not confirmed in the plan)',
    '2605.00236': 'ARA (not confirmed in the plan)',
    '2410.13708': 'Sahara (not confirmed in the plan)',
    '2509.25843': 'ASGuard (not confirmed in the plan)',
}
UNCONF = ('2605.24659', '2608.08027', '2605.00236', '2410.13708', '2509.25843')
fp = f'{OUT}/literature.json'; LIT = json.load(open(fp)) if os.path.exists(fp) else {}
def fetch_abs(aid):
    url = f'https://export.arxiv.org/api/query?id_list={aid}'
    with urllib.request.urlopen(url, timeout=60) as r: x = r.read().decode('utf-8', 'replace')
    e = x.split('<entry>')[-1]
    g = lambda tag: re.sub(r'\s+', ' ', re.search(rf'<{tag}>(.*?)</{tag}>', e, re.S).group(1)).strip() if re.search(rf'<{tag}>(.*?)</{tag}>', e, re.S) else ''
    return dict(title=g('title'), published=g('published'), summary=g('summary'))
for aid in PAPERS:
    if aid in LIT: continue
    try: LIT[aid] = fetch_abs(aid); LIT[aid]['fetched'] = True
    except Exception as e_: LIT[aid] = dict(title='', published='', summary='', fetched=False, error=f'{type(e_).__name__}: {e_}')
    json.dump(LIT, open(fp, 'w'))
rows = []
for aid, note in PAPERS.items():
    L = LIT[aid]; rows.append(dict(arxiv=aid, plan_note=note, opened_here=bool(L.get('fetched')), published=L.get('published', '')[:10], title=L.get('title', '')[:110]))
LITT = pd.DataFrame(rows); LITT.to_csv(f'{REP}/literature.csv', index=False)
print(LITT.to_string(index=False))
print('\n=== abstracts of the five the plan could not confirm (read these before citing any of them) ===')
for aid in UNCONF:
    L = LIT[aid]
    print(f"\n--- arXiv {aid} ---\n{'NOT FETCHED: ' + L.get('error', '') if not L.get('fetched') else L['title'] + ' (' + L.get('published', '')[:10] + ')'}")
    if L.get('fetched'): print(L['summary'][:1400])
DROP = [a for a in UNCONF if not LIT[a].get('fetched')]
print('\nunconfirmed and still not opened, so not cited anywhere in this work:', DROP or 'none')
print('Note: the OpenReview item (5h9mS87Pyt) has no arXiv id and is not fetched here; it stays uncited.')

## Stage B. `src/head_patching.py`: per-head patching at the template positions, and the margin

The module only adds hooks and two helpers. It reads `task_state` for the chat template, the suffix check and the encoder, so a prompt is built exactly as every earlier notebook built it.

In [ ]:
%%writefile src/head_patching.py
"""Per-head patching at the template positions, for the carrier-heads notebooks. Built on task_state, which is unchanged.

A query head's output is the slice of the input to its layer's attention output projection (o_proj) that belongs to that
head. These hooks sit on o_proj as forward pre-hooks, so they see that input before the projection mixes the heads, and
can record or replace any subset of head slices at the K template positions of a prompt pass. Document positions are
never touched and decode steps (sequence length one) pass through.

spec: {layer: dict(kind, heads, src)}
  kind 'set'    the listed heads' slices <- src
  kind 'zero'   the listed heads' slices <- 0
  src  [B, K, nh*dh] (per row), [K, nh*dh] (shared over rows) or [nh*dh] (shared over rows and positions); only the
       listed heads' slices are read from it, so a full recorded tensor can be passed and the heads chosen per call.

offset: 0, or the number of tokens appended after the template (16 when answer tokens are appended for the margin).
"""
from __future__ import annotations
import numpy as np
from task_state import encode, _generate, decoder_layers


def head_dims(model):
    """(number of query heads, head dimension) read from o_proj, so grouped-query attention is handled."""
    m = model.get_base_model() if hasattr(model, "get_base_model") else model
    nh = int(m.config.num_attention_heads); din = int(decoder_layers(model)[0].self_attn.o_proj.in_features)
    assert din % nh == 0, (din, nh)
    return nh, din // nh


class HeadHooks:
    def __init__(self, model, K):
        self.layers = decoder_layers(model); self.K = K; self.offset = 0
        self.nh, self.dh = head_dims(model)
        self.record = False; self.rec = {}; self.spec = {}; self.prompt_passes = 0
        self.handles = [l.self_attn.o_proj.register_forward_pre_hook(self._make(i)) for i, l in enumerate(self.layers)]

    def n_heads(self):
        return len(self.layers) * self.nh

    def _slice(self):
        return slice(-self.K - self.offset, None if self.offset == 0 else -self.offset)

    def _make(self, i):
        def hook(module, args):
            import torch
            x = args[0]
            if not isinstance(x, torch.Tensor) or x.dim() != 3 or x.shape[1] <= 1: return None
            if i == 0: self.prompt_passes += 1
            sl = self._slice()
            if self.record: self.rec[i] = x[:, sl, :].detach().half().cpu().clone()
            s = self.spec.get(i)
            if s is None: return None
            B = x.shape[0]; x = x.clone()
            cur = x[:, sl, :].reshape(B, self.K, self.nh, self.dh); new = cur.clone()
            idx = torch.as_tensor(list(s["heads"]), device=x.device, dtype=torch.long)
            if s["kind"] == "zero":
                new[:, :, idx, :] = 0
            elif s["kind"] == "set":
                v = s["src"].to(x.device, x.dtype)
                if v.dim() == 1: v = v.view(1, 1, self.nh, self.dh).expand(B, self.K, -1, -1)
                elif v.dim() == 2: v = v.reshape(1, self.K, self.nh, self.dh).expand(B, -1, -1, -1)
                elif v.dim() == 3: v = v.reshape(v.shape[0], self.K, self.nh, self.dh)
                else: raise ValueError(f"src has {v.dim()} dimensions")
                new[:, :, idx, :] = v[:, :, idx, :]
            else: raise ValueError(s["kind"])
            x[:, sl, :] = new.reshape(B, self.K, self.nh * self.dh)
            return (x,) + tuple(args[1:])
        return hook

    def close(self):
        for h in self.handles: h.remove()
        self.handles = []


def head_spec(layer, heads, src=None, kind="set"):
    """One spec entry. src is a stacked recording [B, L, K, nh*dh]; its layer slice is taken here."""
    e = dict(kind=kind, heads=list(heads))
    if kind != "zero":
        assert src is not None, "kind 'set' needs a source"
        e["src"] = src[:, int(layer)] if src.dim() == 4 else src
    return {int(layer): e}


def merge_specs(*specs):
    """Specs on disjoint layers merged; two entries for one layer are joined when their kind and source match."""
    out = {}
    for s in specs:
        for l, e in s.items():
            if l not in out: out[l] = dict(e)
            else:
                assert out[l]["kind"] == e["kind"], f"two kinds on layer {l}"
                out[l]["heads"] = sorted(set(out[l]["heads"]) | set(e["heads"]))
    return out


def set_spec(heads, src=None, kind="set"):
    """A spec for a set of (layer, head) pairs."""
    by = {}
    for l, h in heads: by.setdefault(int(l), []).append(int(h))
    return merge_specs(*[head_spec(l, hs, src, kind) for l, hs in by.items()])


def capture_heads(model, hooks, tok, prompts, system, sfx):
    """Head outputs at the template positions for one batch: float16 [B, L, K, nh*dh], from the answering call's prompt pass."""
    import torch
    enc = encode(tok, prompts, system, sfx).to(model.device)
    hooks.rec = {}; hooks.spec = {}; hooks.offset = 0; hooks.record = True
    try: _generate(model, tok, enc, 1)
    finally: hooks.record = False
    st = torch.stack([hooks.rec[i] for i in range(len(hooks.layers))], 1); hooks.rec = {}
    return st


def answer_logprob(model, hooks, tok, prompts, system, sfx, ans_ids, spec=None):
    """Mean log-probability of each row's answer tokens, with the head spec applied at the template positions.

    ans_ids: one list of token ids per row, all the same length. The answer tokens are appended to the prompt, so the
    template positions sit len(ans) places before the end and the hooks take that offset."""
    import torch
    enc = encode(tok, prompts, system, sfx).to(model.device)
    A = torch.as_tensor(ans_ids, device=enc["input_ids"].device, dtype=enc["input_ids"].dtype)
    assert A.dim() == 2 and A.shape[0] == enc["input_ids"].shape[0], (A.shape, enc["input_ids"].shape)
    ids = torch.cat([enc["input_ids"], A], 1); am = torch.cat([enc["attention_mask"], torch.ones_like(A)], 1)
    nt = int(A.shape[1])
    hooks.rec = {}; hooks.spec = dict(spec or {}); hooks.record = False; hooks.offset = nt
    try:
        with torch.no_grad(): lg = model(input_ids=ids, attention_mask=am).logits
    finally: hooks.spec = {}; hooks.offset = 0
    lp = torch.log_softmax(lg[:, -nt - 1:-1, :].float(), -1)
    return lp.gather(-1, A[:, :, None].long()).squeeze(-1).mean(1).detach().cpu().numpy()


def doc_attention(model, tok, prompts, system, sfx, doc_tokens, batch=4):
    """Attention mass from the last prompt position to the document span, per head: [L, nh] averaged over rows.

    Attention Tracker ranks heads by the mass the last token sends to the injected instruction. That span is not
    recorded for rewritten documents, so the span used here is the whole document, which is the closest recoverable
    form of the rule. doc_tokens: the document's own token count per row."""
    import torch
    L = len(decoder_layers(model)); nh, _ = head_dims(model); tot = np.zeros((L, nh)); rows = 0
    for s0 in range(0, len(prompts), batch):
        ps = prompts[s0:s0 + batch]; nd = doc_tokens[s0:s0 + batch]
        enc = encode(tok, ps, system, sfx).to(model.device)
        with torch.no_grad(): att = model(**enc, output_attentions=True).attentions
        T = enc["input_ids"].shape[1]; K = len(sfx)
        for b in range(len(ps)):
            lo = max(0, T - K - int(nd[b])); hi = T - K
            if hi <= lo: continue
            for l in range(L): tot[l] += att[l][b, :, -1, lo:hi].float().sum(-1).cpu().numpy()
            rows += 1
        del att
    assert rows, "no row had a document span"
    return tot / rows

## Stage C. The items of notebook 53, rebuilt from its caches; the model; the hook self-test

The 72 robust successes, 71 robust failures and the clean twins come from notebook 53's own caches and are checked against its committed per-item file. Nothing is screened again and no answer is regenerated here.

In [ ]:
import importlib, hashlib, collections, numpy as np
import task_state, state_patching, head_patching, adaptive_attack, adaptive_strict, final_scoring, targets, attack_success, ci_tools, fidelity_audit, model_defense
for m_ in (task_state, state_patching, head_patching, adaptive_attack, adaptive_strict, final_scoring, targets, attack_success, ci_tools, fidelity_audit, model_defense): importlib.reload(m_)
from task_state import suffix_ids, token_f1, degenerate, strat_auroc, strat_auroc_boot
from state_patching import PatchHooks, layer_spec, generate_patched, capture_states
from head_patching import HeadHooks, capture_heads, answer_logprob, set_spec, doc_attention, head_dims
from adaptive_attack import host_of
from adaptive_strict import load_attempts
from final_scoring import final_rule, paper_verdict, langdetect_non_english, CATEGORIES, JUDGE_ONLY, TRANSLATION
from targets import load_lm, generate_batch, TASK_SYSTEM, TASK_USER
from attack_success import judge_prompt, parse_judge, JUDGE_SYSTEM
from model_defense import TASK_USER_WITH
from ci_tools import wilson
from fidelity_audit import newcombe_paired
H = lambda s: hashlib.sha1((s or '').encode()).hexdigest()
def jload(f): return json.load(open(f)) if os.path.exists(f) else {}
def jsave(obj, f): json.dump(obj, open(f + '.tmp', 'w')); os.replace(f + '.tmp', f)
pd.set_option('display.width', 260); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 90)

# ---- settings and marks, fixed before any result of this notebook ----
SEED = 56
TARGET, JUDGE = 'Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct'
MODELS = {'qwen3b_undefended': dict(adapter=None, label='Qwen 3B undefended')}; U = 'qwen3b_undefended'; LB = {t: MODELS[t]['label'] for t in MODELS}
BS, MAX_NEW, NTOK = 16, 160, 16
BS_SWEEP = 16                      # items per sweep forward pass (two rows per item)
KS = (10, 20, 40); KMAIN = 20
N_SWEEP = 32                       # robust items the sweep runs on; the set checks use all 72
N_INSTR_SWEEP = 30                 # items of pool one the instruction sweep runs on (six per family)
RAND_DRAWS = 3
CRIT = dict(proxy_auroc=0.80, gate2_auroc=0.75,
            l1_suff=0.30, l1_suff_partial=0.15, l1_ctrl=0.05,
            l2_stop=0.60, l2_ctrl=0.15, l2_paired_lo=0.25,
            l3_f1_ref=0.597, l3_f1_tol=0.10, l3_degen=0.05,
            i1_do=0.30, o1_shared_heads=8, o1_rho=0.40, o1_cross=0.30, o1_sep_heads=2, o1_sep_cross=0.10)
FAMILIES = {'Language Translation': 'translation', 'Emoji Substitution': 'emoji', 'Base Encoding': 'base64', 'Reverse Text': 'reverse', 'Substitution Ciphers': 'cipher'}
NB51, NB53 = 'data/task_state', 'data/state_patching'
for t in MODELS: os.makedirs(f'{OUT}/{t}', exist_ok=True)
for f in ('clean_out.json', 'judge.json', 'p2_base.json', 'p3_base.json'): assert os.path.exists(f'{NB51}/{U}/{f}'), f'notebook 51 cache missing: {f}'
for f in ('screen.json', 'judge.json', 'twin_donor.json'): assert os.path.exists(f'{NB53}/{U}/{f}'), f'notebook 53 cache missing: {f}'

# ---- the 210 injections and the clean twins, as notebooks 51 and 53 built them ----
SUB = pd.DataFrame(json.load(open('data/phase5v3/subset_full.json'))); n = len(SUB); assert set(SUB.meta) == set(CATEGORIES)
SIDX = [i for i in range(n) if SUB.meta.iloc[i] not in JUDGE_ONLY]; JIDX = [i for i in range(n) if SUB.meta.iloc[i] in JUDGE_ONLY]
SPLIT = [host_of(SUB.text.iloc[i], SUB.instruction.iloc[i]) for i in range(n)]
CLEAN_DOC = [(b[:-1] if b.endswith('\n') else b) + a for b, a in SPLIT]; CLEAN_P = [TASK_USER.format(doc=d) for d in CLEAN_DOC]
assert open(f'{NB51}/{U}/clean_docs.sha1').read().strip() == H('\x00'.join(CLEAN_DOC)), 'the clean twins differ from notebook 51\'s'
LD = {}
def ld(o):
    k = H(o or '')
    if k not in LD: LD[k] = langdetect_non_english(o)
    return LD[k]
JC = {**jload(f'{NB51}/{U}/judge.json'), **jload(f'{NB53}/{U}/judge.json'), **jload(f'{OUT}/{U}/judge.json')}
jkey = lambda goal, out: H(goal + '\x00' + (out or ''))
def needs_judge(cat, out, doc): return (not ld(out)) if cat == TRANSLATION else final_rule(cat, out, doc) is None
def ok(cat, goal, doc, o):
    """The paper's verdict: did answer o follow instruction `goal` of category `cat` in document `doc`."""
    k = jkey(goal, o)
    if needs_judge(cat, o, doc):
        assert k in JC, 'an answer the rule sends to the judge was not judged'
        return bool(paper_verdict(cat, o, doc, JC[k]))
    return bool(paper_verdict(cat, o, doc, False))

# ---- notebook 53's items: the first successful attempt per injection, and the failed originals ----
def attempts():
    rows = []
    for log, idx in ((load_attempts('data/adaptive_strict', U, 4), SIDX), (load_attempts('data/final_scoring/adaptive', U, 4), JIDX)):
        for a in log:
            i = idx[a['i']]; cat = SUB.meta.iloc[i]
            rows.append(dict(injection=i, category=cat, goal=SUB.instruction.iloc[i], attempt=int(a['attempt']), document=a['document'], output=a['output'],
                             success=bool(a['verdict']) or (cat == TRANSLATION and ld(a['output'])), key=H(f"{i}\x00{a['document']}")))
    return pd.DataFrame(rows).sort_values(['injection', 'attempt']).reset_index(drop=True)
ATT = attempts()
P2B, P3B, CO = jload(f'{NB51}/{U}/p2_base.json'), jload(f'{NB51}/{U}/p3_base.json'), jload(f'{NB51}/{U}/clean_out.json')
first = ATT[ATT.success].groupby('injection').head(1)
SUCC = [dict(injection=int(r.injection), category=r.category, goal=r.goal, document=r.document, attempt=int(r.attempt), key=r.key, prompt=TASK_USER.format(doc=r.document)) for r in first.itertuples()]
FAIL = [dict(injection=int(r.injection), category=r.category, goal=r.goal, document=r.document, attempt=1, key=r.key, prompt=TASK_USER.format(doc=r.document)) for r in ATT[(ATT.attempt == 1) & ~ATT.success].itertuples()]
for it in SUCC: it['greedy'] = P2B[it['key']][0]; it['greedy_ok'] = ok(it['category'], it['goal'], it['document'], it['greedy'])
for it in FAIL: it['greedy'] = P3B[it['key']][0]; it['greedy_ok'] = ok(it['category'], it['goal'], it['document'], it['greedy'])
SUCC = [it for it in SUCC if it['greedy_ok']]; FAIL = [it for it in FAIL if not it['greedy_ok']]
assert len(SUCC) == 147 and len(FAIL) == 119, (len(SUCC), len(FAIL))
for it in SUCC + FAIL: it['first_clean'] = int(CO[str(it['injection'])]['first']); it['clean_answer'] = CO[str(it['injection'])]['output']
TWIN = [dict(injection=it['injection'], category=it['category'], goal=it['goal'], document=CLEAN_DOC[it['injection']], key=f"twin{it['injection']}", prompt=CLEAN_P[it['injection']],
             greedy=CO[str(it['injection'])]['output'], first=int(CO[str(it['injection'])]['first'])) for it in SUCC]

# ---- notebook 53's five-sample screen, from its cache; the robust sets must match its committed file ----
SCR = jload(f'{NB53}/{U}/screen.json')
for it in SUCC + FAIL:
    assert it['key'] in SCR, 'notebook 53 screen cache incomplete'
    oks = [ok(it['category'], it['goal'], it['document'], o) for o in SCR[it['key']]]
    it['samples_ok'] = int(sum(oks)); it['robust'] = (it['greedy_ok'] and it['samples_ok'] >= 4) or ((not it['greedy_ok']) and it['samples_ok'] == 0)
RS = [it for it in SUCC if it['robust']]; RF = [it for it in FAIL if it['robust']]
S53 = pd.read_csv('reports/state_patching/screen_items.csv'); s53 = S53[S53.model == U]
assert len(RS) == int((s53.label == 'robust success').sum()) == 72, (len(RS), int((s53.label == 'robust success').sum()))
assert len(RF) == int((s53.label == 'robust failure').sum()) == 71, (len(RF), int((s53.label == 'robust failure').sum()))
RSI = {it['injection'] for it in RS}; TWIN_RS = [tw for tw in TWIN if tw['injection'] in RSI]; assert len(TWIN_RS) == 72
BYKEY = {it['key']: it for it in SUCC}; BYINJ = {it['injection']: it for it in RS}
print('items:', dict(successes=len(SUCC), robust_successes=len(RS), failed_originals=len(FAIL), robust_failures=len(RF), twins_of_robust=len(TWIN_RS)))
print('robust successes by family:', dict(collections.Counter(it['category'] for it in RS)))

# ---- notebook 53's donors, replicated in its own order so its cached transplant answers line up ----
rng53 = np.random.default_rng(53)
def pick(c): return c[int(rng53.integers(len(c)))] if c else None
for it in SUCC:
    same = lambda d: d['category'] == it['category'] and d['injection'] != it['injection'] and d['document'] != it['document']
    for cand in ([d for d in RS if same(d)], [d for d in SUCC if same(d)], [d for d in RS if d['category'] != it['category']]):
        if cand: it['ctrlA'] = pick(cand)['key']; break
    else: it['ctrlA'] = None
    for cand in ([d for d in RF if same(d)], [d for d in FAIL if same(d)], [d for d in RF if d['category'] != it['category']]):
        if cand: it['ctrlB'] = pick(cand)['key']; break
    else: it['ctrlB'] = None
for grp in (FAIL, TWIN):
    for it in grp:
        d = pick([x for x in RS if x['category'] != it['category'] and x['injection'] != it['injection']])
        it['donor'] = d['key'] if d else None; it['donor_goal'] = d['goal'] if d else None; it['donor_category'] = d['category'] if d else None
for grp in (SUCC, FAIL, TWIN):
    k_ = len(grp)
    if k_ >= 2:
        while True:
            p = rng53.permutation(k_)
            if not (p == np.arange(k_)).any(): break
        for a, b in enumerate(p): grp[a]['otherclean'] = f"twin{grp[b]['injection']}"
TD = jload(f'{NB53}/{U}/twin_donor.json')
assert all(it['key'] in TD for it in TWIN if it['donor']), 'the donor assignment does not match notebook 53\'s cached transplant answers'
print('notebook 53 donors replicated and checked against its cached transplant answers:', len([it for it in TWIN if it["donor"]]), 'twins')

# ---- the model, the judge, and the hook self-test ----
SFX = None; _M = {}
def use():
    global SFX
    if _M: return _M['model'], _M['tok'], _M['hk'], _M['ph']
    m, tok = load_lm(TARGET, four_bit=True); s = suffix_ids(tok, TASK_SYSTEM); SFX = s
    nh, dh = head_dims(m); _M.update(model=m, tok=tok, hk=HeadHooks(m, len(s)), ph=PatchHooks(m, len(s)))
    print(f'loaded {LB[U]}: {len(_M["hk"].layers)} layers x {nh} query heads = {_M["hk"].n_heads()} heads, head dimension {dh}; task state = the last {len(s)} prompt tokens {tok.convert_ids_to_tokens(s)}')
    return m, tok, _M['hk'], _M['ph']
_J = {}
def seven_b():
    if 'm' not in _J: _J['m'], _J['t'] = load_lm(JUDGE, four_bit=True)
    return _J['m'], _J['t']
def free_seven_b():
    if 'm' in _J: _J.clear(); gc.collect(); torch.cuda.empty_cache()
def judge_pending(triples):
    need = {}
    for cat, goal, doc, o in triples:
        k = jkey(goal, o)
        if k not in JC and k not in need and needs_judge(cat, o, doc): need[k] = (goal, o)
    if need:
        jm, jt = seven_b(); ks = list(need); own = jload(f'{OUT}/{U}/judge.json')
        for s in range(0, len(ks), 400):
            ch = ks[s: s + 400]; jo = generate_batch(jm, jt, [judge_prompt(*need[k]) for k in ch], system=JUDGE_SYSTEM, max_new_tokens=8, batch_size=8, temperature=0.0, show_every=400)
            for k, x in zip(ch, jo): JC[k] = own[k] = bool(parse_judge(x))
            jsave(own, f'{OUT}/{U}/judge.json')
    return len(need)
by_len = lambda ps: sorted(range(len(ps)), key=lambda i: (len(ps[i]), i))
chunks = lambda order, bs: [order[s: s + bs] for s in range(0, len(order), bs)]
ans_ids = lambda tok, s: tok(s or ' ', add_special_tokens=False).input_ids[:NTOK]
def pad_ans(tok, s):
    """The first NTOK tokens of an answer, padded with the end-of-text token so every row has the same length."""
    v = ans_ids(tok, s); return v + [tok.pad_token_id] * (NTOK - len(v))
def rate_(x):
    x = np.asarray(x, bool); k_, m_ = int(x.sum()), len(x); lo, hi = wilson(k_, m_) if m_ else (np.nan, np.nan)
    return f'{k_}/{m_} = {k_ / m_:.3f} [{lo:.3f}, {hi:.3f}]' if m_ else 'no items'
m, tok, hk, ph = use()
def self_test():
    its = RS[:8]; ps = [it['prompt'] for it in its]
    Aa = capture_heads(m, hk, tok, ps, TASK_SYSTEM, SFX); before = hk.prompt_passes
    o1, _ = generate_patched(m, ph, tok, ps, TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW); passes = hk.prompt_passes - before
    allh = [(l, h) for l in range(len(hk.layers)) for h in range(hk.nh)]
    o2, _ = generate_patched(m, ph, tok, ps, TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW)   # same call, no spec: the run-to-run floor
    hk.spec = set_spec(allh, Aa); o3, _ = generate_patched(m, ph, tok, ps, TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW); hk.spec = {}
    same = sum(a == b for a, b in zip(o1, o3)); floor = sum(a == b for a, b in zip(o1, o2))
    lp = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, [pad_ans(tok, it['greedy']) for it in its])
    lpc = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, [pad_ans(tok, it['clean_answer']) for it in its])
    marg = lp - lpc
    nz = {}
    for l in (0, len(hk.layers) // 2, len(hk.layers) - 1):
        z = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, [pad_ans(tok, it['greedy']) for it in its], spec=set_spec([(l, 0)], None, kind='zero'))
        nz[l] = float(np.abs(z - lp).max())
    zc = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, [pad_ans(tok, it['greedy']) for it in its], spec=set_spec(allh, Aa))
    own = float(np.abs(zc - lp).max())
    print(f'self-test: prompt passes per call {passes}; answers unchanged when every head output is written back {same}/{len(o1)} (no-spec repeat {floor}/{len(o1)}); '
          f'injection margin on the attacked runs, median {float(np.median(marg)):+.3f}, min {float(marg.min()):+.3f}; '
          f'writing every head output back changes the margin by at most {own:.2e}; zeroing one head moves the margin by at most {nz}')
    assert passes == 1, 'more than one prompt pass per call'
    assert same >= floor - 1, 'writing a head output back unchanged altered the answers: the hooks are wrong'
    assert float(np.median(marg)) > 0, 'the attacked answer is not more likely than the clean answer on attacked prompts'
    assert own < 1e-2, 'writing every head output back changed the margin: the recording and the write do not line up'
    assert max(nz.values()) > 0, 'zeroing a head changed nothing: the hooks are not reaching the model'
self_test()

## Stage D. Gate one: does the injection margin agree with the judge on the full-state effect?

Three versions of the same question, all on the full residual state at the template positions, so the measure is tested on an effect large enough to judge.

**A, the gate (same document).** The attacked run's whole template state written into its own clean twin. This is the full-state version of exactly what the sufficiency sweep does head by head, so its answer pair is the sweep's answer pair. Notebook 53 never ran it; the 72 answers are generated and judged here.

**B, reported.** The clean twin's state written into the attacked run, from notebook 53's cache: the necessity direction, same answer pair, judged there.

**C, reported, and this is the check that failed as the plan wrote it.** Notebook 53's transplant, whose donor comes from another category and another document, so the donor's answer text is about a document the run never saw. Its number is printed and kept.

In [ ]:
def auroc_ci(score, label, stratum, cluster):
    a = strat_auroc(score, label, stratum); lo, hi = strat_auroc_boot(score, label, stratum, cluster, B=2000, seed=SEED); return a, lo, hi
ST = {}
def states_of(keys_prompts):
    """Residual template states (notebook 53's level), cached in memory."""
    need = []
    for k, p in keys_prompts:
        if k not in ST and k not in [x[0] for x in need]: need.append((k, p))
    for idx in chunks(by_len([p for _, p in need]), BS):
        st = capture_states(m, ph, tok, [need[i][1] for i in idx], TASK_SYSTEM, SFX)
        for j, i in enumerate(idx): ST[need[i][0]] = st[j]
    return ST
def margin_of(items, prompt_of, src_of, ans_a, ans_b, fp, hooks):
    """Injection margin per item: two rows of one batch, the 16 answer tokens appended, the full state written."""
    C_ = jload(fp); todo = [it for it in items if it['key'] not in C_]
    for bi, idx in enumerate(chunks(by_len([prompt_of(it) for it in todo]), BS_SWEEP)):
        bx = [todo[i] for i in idx]
        ps = [p for it in bx for p in (prompt_of(it), prompt_of(it))]
        aid = [v for it in bx for v in (pad_ans(tok, ans_a(it)), pad_ans(tok, ans_b(it)))]
        src = torch.stack([src_of(it) for it in bx for _ in range(2)])
        lp = answer_logprob(m, hooks, tok, ps, TASK_SYSTEM, SFX, aid, spec=layer_spec(range(src.shape[1]), 'mix', src, 1.0))
        for j, it in enumerate(bx): C_[it['key']] = float(lp[2 * j] - lp[2 * j + 1])
        jsave(C_, fp); print(f'  margin: {min((bi + 1) * BS_SWEEP, len(todo))}/{len(todo)} new runs scored')
    return C_
GROWS = []

# ---- A. the gate: the attacked run's full template state written into its own clean twin (same document) ----
states_of([(tw['key'], tw['prompt']) for tw in TWIN_RS] + [(BYINJ[tw['injection']]['key'], BYINJ[tw['injection']]['prompt']) for tw in TWIN_RS])
fp = f'{OUT}/{U}/gen_full_suff.json'; FS = jload(fp); todo = [tw for tw in TWIN_RS if tw['key'] not in FS]
for bi, idx in enumerate(chunks(by_len([tw['prompt'] for tw in todo]), BS)):
    bx = [todo[i] for i in idx]; src = torch.stack([ST[BYINJ[tw['injection']]['key']] for tw in bx])
    outs, _ = generate_patched(m, ph, tok, [tw['prompt'] for tw in bx], TASK_SYSTEM, SFX, spec=layer_spec(range(src.shape[1]), 'mix', src, 1.0), max_new_tokens=MAX_NEW)
    FS.update({tw['key']: [o] for tw, o in zip(bx, outs)}); jsave(FS, fp); print(f'  full-state sufficiency: {min((bi + 1) * BS, len(todo))}/{len(todo)} new answers generated')
src_of = lambda tw: BYINJ[tw['injection']]
judge_pending([(src_of(tw)['category'], src_of(tw)['goal'], tw['document'], FS[tw['key']][0]) for tw in TWIN_RS]); free_seven_b()
y_a = np.array([ok(src_of(tw)['category'], src_of(tw)['goal'], tw['document'], FS[tw['key']][0]) for tw in TWIN_RS], bool)
MA = margin_of(TWIN_RS, lambda tw: tw['prompt'], lambda tw: ST[src_of(tw)['key']], lambda tw: src_of(tw)['greedy'], lambda tw: tw['greedy'], f'{OUT}/{U}/margin_full_suff.json', ph)
m_a = np.array([MA[tw['key']] for tw in TWIN_RS]); c_a = np.array([src_of(tw)['category'] for tw in TWIN_RS]); k_a = np.array([tw['injection'] for tw in TWIN_RS])
A_BOTH = bool(0 < y_a.sum() < len(y_a))
a_a, lo_a, hi_a = auroc_ci(m_a, y_a, c_a, k_a) if A_BOTH else (float('nan'),) * 3
print(f'\nA, the gate (same document): the full state of the attacked run written into its own clean twin makes the twin follow the injection in {rate_(y_a)}'
      + (f'; AUROC of the margin against the judge = {a_a:.3f} [{lo_a:.3f}, {hi_a:.3f}]' if A_BOTH
         else '; every run fell on one side of the judge, so an AUROC cannot be computed here and the gate falls to version B, which is the same answer pair in the other direction'))
GROWS.append(dict(version='A, same document' + (' (the gate)' if A_BOTH else ' (one class only: cannot be the gate)'), what='attacked run\'s full template state into its own clean twin, 72 twins',
                  n=len(TWIN_RS), followed=int(y_a.sum()), auroc=round(a_a, 3) if A_BOTH else np.nan, lo=round(lo_a, 3) if A_BOTH else np.nan, hi=round(hi_a, 3) if A_BOTH else np.nan,
                  mark=CRIT['proxy_auroc'] if A_BOTH else np.nan, met=bool(A_BOTH and a_a >= CRIT['proxy_auroc']) if A_BOTH else np.nan))

# ---- B. reported: the necessity direction, from notebook 53's cached answers ----
NBC = jload(f'{NB53}/{U}/succ_clean_L0-31.json'); HAVE_B = bool(NBC) and all(it['key'] in NBC for it in RS); a_b = lo_b = hi_b = float('nan'); B_BOTH = False
if HAVE_B:
    y_b = np.array([not ok(it['category'], it['goal'], it['document'], NBC[it['key']][0]) for it in RS], bool)
    MB = margin_of(RS, lambda it: it['prompt'], lambda it: ST[f"twin{it['injection']}"], lambda it: it['greedy'], lambda it: it['clean_answer'], f'{OUT}/{U}/margin_full_nec.json', ph)
    m_b = -np.array([MB[it['key']] for it in RS]); B_BOTH = bool(0 < y_b.sum() < len(y_b))
    if B_BOTH: a_b, lo_b, hi_b = auroc_ci(m_b, y_b, np.array([it['category'] for it in RS]), np.array([it['injection'] for it in RS]))
    print(f'B (necessity, notebook 53\'s cache): the clean twin\'s full state written into the attacked run stops {rate_(y_b)}'
          + (f'; AUROC of the margin, sign reversed, = {a_b:.3f} [{lo_b:.3f}, {hi_b:.3f}]' if B_BOTH else '; one class only'))
    GROWS.append(dict(version='B, necessity' + ('' if A_BOTH else ' (the gate, A had one class only)'), what='clean twin\'s full state into the attacked run, notebook 53\'s judged answers, 72 items',
                      n=len(RS), followed=int(y_b.sum()), auroc=round(a_b, 3) if B_BOTH else np.nan, lo=round(lo_b, 3) if B_BOTH else np.nan, hi=round(hi_b, 3) if B_BOTH else np.nan,
                      mark=np.nan if A_BOTH else CRIT['proxy_auroc'], met=np.nan if A_BOTH else bool(B_BOTH and a_b >= CRIT['proxy_auroc'])))
else:
    print('B, reported: notebook 53\'s succ_clean_L0-31 answers are not all in its cache, so the necessity version is not computed')

# ---- C. reported: the gate as the plan wrote it, whose donor comes from another document ----
states_of([(it['key'], it['prompt']) for it in TWIN if it['donor']] + [(BYKEY[it['donor']]['key'], BYKEY[it['donor']]['prompt']) for it in TWIN if it['donor']])
TWD = [it for it in TWIN if it['donor'] and it['key'] in TD]
judge_pending([(it['donor_category'], it['donor_goal'], it['document'], TD[it['key']][0]) for it in TWD]); free_seven_b()
for it in TWD: it['donor_followed'] = ok(it['donor_category'], it['donor_goal'], it['document'], TD[it['key']][0])
y_c = np.array([it['donor_followed'] for it in TWD], bool)
MC = margin_of(TWD, lambda it: it['prompt'], lambda it: ST[it['donor']], lambda it: BYKEY[it['donor']]['greedy'], lambda it: it['greedy'], f'{OUT}/{U}/margin_transplant.json', ph)
m_c = np.array([MC[it['key']] for it in TWD]); a_c, lo_c, hi_c = auroc_ci(m_c, y_c, np.array([it['donor_category'] for it in TWD]), np.array([it['injection'] for it in TWD]))
print(f'C, reported (the gate as the plan wrote it, cross-document donor): the donor\'s injection was followed in {rate_(y_c)} (notebook 53 reported 0.565); '
      f'AUROC of the margin against the judge = {a_c:.3f} [{lo_c:.3f}, {hi_c:.3f}]; this is the check that failed, and the reason is that the donor\'s answer is about another document')
GROWS.append(dict(version='C, cross-document transplant (the gate as pre-registered; FAILED)', what='notebook 53\'s transplant, donor from another category and document', n=len(TWD), followed=int(y_c.sum()),
                  auroc=round(a_c, 3), lo=round(lo_c, 3), hi=round(hi_c, 3), mark=CRIT['proxy_auroc'], met=bool(a_c >= CRIT['proxy_auroc'])))
# ---- which version is the gate: A when it has both classes, otherwise B ----
GATE_NAME = 'A, same document' if A_BOTH else ('B, necessity (A had one class only)' if B_BOTH else 'none available')
PROXY, plo, phi = (a_a, lo_a, hi_a) if A_BOTH else ((a_b, lo_b, hi_b) if B_BOTH else (float('nan'),) * 3)
PROXY_OK = bool(PROXY == PROXY and PROXY >= CRIT['proxy_auroc'])
GT1 = pd.DataFrame(GROWS); GT1.to_csv(f'{REP}/proxy_gates.csv', index=False)
print('\n=== gate one, every version computed ==='); print(GT1.to_string(index=False))
print(f'the gate is version "{GATE_NAME}": AUROC {PROXY:.3f}, mark {CRIT["proxy_auroc"]:.2f} -> {"pass" if PROXY_OK else "FAIL"}')
if not PROXY_OK: print('\nSTOP: the margin does not track the judge on the same-document full-state effect. The sweep is not run and the plan\'s generation fallback is needed.')

## Stage E. The two per-head sweeps on 32 robust items

Sufficiency writes each head's output from the attacked run into the clean twin's run; necessity writes the clean twin's into the attacked run. Each head is one forward pass over the batch, two rows per item. Effects are differences from the unpatched margin of the same run, averaged over items, and are cached per head so a disconnected session resumes.

In [ ]:
assert PROXY_OK, 'gate one (same document) failed: do not run the sweep'
rng = np.random.default_rng(SEED)
fam_counts = collections.Counter(it['category'] for it in RS)
def stratified(items, k, seed):
    """k items drawn in proportion to family, deterministic, with at least one from each family that has any."""
    r_ = np.random.default_rng(seed); by = collections.defaultdict(list)
    for it in items: by[it['category']].append(it)
    picks = []
    for c in sorted(by):
        q = max(1, int(round(k * len(by[c]) / len(items)))); idx = r_.permutation(len(by[c]))[:min(q, len(by[c]))]
        picks += [by[c][i] for i in idx]
    r_.shuffle(picks); return picks[:k]
SWEEP_ITEMS = stratified(RS, N_SWEEP, SEED)
print('sweep items:', len(SWEEP_ITEMS), dict(collections.Counter(it['category'] for it in SWEEP_ITEMS)))
HEADS = [(l, h) for l in range(len(hk.layers)) for h in range(hk.nh)]; NH = len(HEADS)
HA = {}
def heads_of(keys_prompts):
    need = [(k, p) for k, p in keys_prompts if k not in HA]
    for idx in chunks(by_len([p for _, p in need]), BS):
        s = capture_heads(m, hk, tok, [need[i][1] for i in idx], TASK_SYSTEM, SFX)
        for j, i in enumerate(idx): HA[need[i][0]] = s[j]
    return HA
heads_of([(it['key'], it['prompt']) for it in SWEEP_ITEMS] + [(f"twin{it['injection']}", CLEAN_P[it['injection']]) for it in SWEEP_ITEMS])
print('head outputs captured for', len(HA), 'runs')
def sweep(name, items, target_prompt, source_key, sign):
    """Per-head mean effect on the injection margin. target_prompt(it) is the run being changed; source_key(it) names the
    run the head output comes from; sign is +1 when a larger margin means more injection following."""
    fp = f'{OUT}/{U}/sweep_{name}.json'; C_ = jload(fp)
    bl = chunks(by_len([target_prompt(it) for it in items]), BS_SWEEP)
    base = C_.get('_baseline')
    if base is None:
        base = {}
        for idx in bl:
            bx = [items[i] for i in idx]
            ps = [p for it in bx for p in (target_prompt(it), target_prompt(it))]
            aid = [v for it in bx for v in (pad_ans(tok, it['greedy']), pad_ans(tok, it['clean_answer']))]
            lp = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, aid)
            for j, it in enumerate(bx): base[it['key']] = float(lp[2 * j] - lp[2 * j + 1])
        C_['_baseline'] = base; jsave(C_, fp)
    print(f'  {name}: unpatched margin, median {float(np.median(list(base.values()))):+.3f}')
    todo = [f'{l}:{h}' for l, h in HEADS if f'{l}:{h}' not in C_]
    for z, hkey in enumerate(todo):
        l, h = (int(x) for x in hkey.split(':')); vals = []
        for idx in bl:
            bx = [items[i] for i in idx]
            ps = [p for it in bx for p in (target_prompt(it), target_prompt(it))]
            aid = [v for it in bx for v in (pad_ans(tok, it['greedy']), pad_ans(tok, it['clean_answer']))]
            src = torch.stack([HA[source_key(it)] for it in bx for _ in range(2)])
            lp = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, aid, spec=set_spec([(l, h)], src))
            vals += [float(lp[2 * j] - lp[2 * j + 1]) - base[it['key']] for j, it in enumerate(bx)]
        C_[hkey] = sign * float(np.mean(vals))
        if z % 50 == 49 or z == len(todo) - 1: jsave(C_, fp); print(f'  {name}: {z + 1}/{len(todo)} new heads swept')
    return C_
SUF = sweep('sufficiency', SWEEP_ITEMS, lambda it: CLEAN_P[it['injection']], lambda it: it['key'], +1)
NEC = sweep('necessity', SWEEP_ITEMS, lambda it: it['prompt'], lambda it: f"twin{it['injection']}", -1)
eff = lambda C_: np.array([C_[f'{l}:{h}'] for l, h in HEADS])
E_SUF, E_NEC = eff(SUF), eff(NEC)
SW = pd.DataFrame([dict(layer=l, head=h, sufficiency=E_SUF[i], necessity=E_NEC[i]) for i, (l, h) in enumerate(HEADS)])
SW.to_csv(f'{REP}/head_effects.csv', index=False)
order = np.argsort(-E_SUF); C_SET = {k: [HEADS[i] for i in order[:k]] for k in KS}
print('\n=== the 20 heads with the largest sufficiency effect ===')
print(SW.iloc[order[:20]].assign(sufficiency=lambda d: d.sufficiency.round(4), necessity=lambda d: d.necessity.round(4)).to_string(index=False))
print('\nlayers of C_20:', dict(collections.Counter(l for l, _ in C_SET[KMAIN])))
print('Spearman correlation of the sufficiency and necessity profiles over all heads:',
      round(float(pd.Series(E_SUF).rank().corr(pd.Series(E_NEC).rank())), 3))

## Stage F. Set checks by generation and the judge, on all 72 robust items

Every claim is checked with the whole set patched at once and an answer generated and judged, against three control sets of the same size. The sweep chose the sets on 32 items; these checks run on all 72.

In [ ]:
def rate(x):
    x = np.asarray(x, bool); k, m_ = int(x.sum()), len(x); lo, hi = wilson(k, m_) if m_ else (np.nan, np.nan); return dict(n=m_, k=k, rate=k / m_ if m_ else np.nan, lo=lo, hi=hi)
rstr = lambda r, d=3: f"{r['rate']:.{d}f} [{r['lo']:.{d}f}, {r['hi']:.{d}f}]" if r['n'] else 'no items'
dstr = lambda d: f'{d[0]:+.2f} [{d[1]:+.2f}, {d[2]:+.2f}]' if not np.isnan(d[0]) else ''
# control sets, fixed here: random matched by layer, the lowest-effect heads within C_k's layers, and the attention-mass set
EI = {hd: i for i, hd in enumerate(HEADS)}
def random_matched(cset, draw):
    r_ = np.random.default_rng(SEED * 100 + draw); by = collections.Counter(l for l, _ in cset); out = []
    for l, q in by.items():
        pool = [h for h in range(hk.nh) if (l, h) not in cset]; out += [(l, int(h)) for h in r_.permutation(pool)[:q]]
    return out
def complement_low(cset):
    lays = {l for l, _ in cset}; pool = [(l, h) for l, h in HEADS if l in lays and (l, h) not in cset]
    return [pool[i] for i in np.argsort([E_SUF[EI[x]] for x in pool])[:len(cset)]]
fp = f'{OUT}/{U}/attention_mass.npy'; AM = np.load(fp) if os.path.exists(fp) else None
if AM is None:
    was = getattr(m.config, '_attn_implementation', None)
    try:
        m.config._attn_implementation = 'eager'      # attention weights are only returned by the eager path
        AI = stratified(RS, 8, SEED + 9)
        AM = doc_attention(m, tok, [it['prompt'] for it in AI], TASK_SYSTEM, SFX,
                           [len(tok(it['document'], add_special_tokens=False).input_ids) for it in AI], batch=1); np.save(fp, AM)
    except Exception as e_:
        print(f'the attention-mass control set could not be computed ({type(e_).__name__}: {e_}); it is dropped and the marks name the controls actually used')
    finally:
        if was is not None: m.config._attn_implementation = was
        gc.collect(); torch.cuda.empty_cache()
ATT_OK = AM is not None
if ATT_OK: AMF = AM.reshape(-1); attention_set = lambda k: [HEADS[i] for i in np.argsort(-AMF)[:k]]
SETS = {f'C_{k}': C_SET[k] for k in KS}
for k in KS:
    for d in range(RAND_DRAWS): SETS[f'random_{k}_draw{d + 1}'] = random_matched(C_SET[k], d + 1)
    SETS[f'complement_{k}'] = complement_low(C_SET[k])
    if ATT_OK: SETS[f'attention_{k}'] = attention_set(k)
SETS['all_heads'] = HEADS
json.dump({k: [[int(l), int(h)] for l, h in v] for k, v in SETS.items()}, open(f'{OUT}/{U}/head_sets.json', 'w'))
print('head sets:', {k: len(v) for k, v in SETS.items()})
heads_of([(it['key'], it['prompt']) for it in RS] + [(f"twin{it['injection']}", CLEAN_P[it['injection']]) for it in RS])
MEANC = torch.stack([HA[f"twin{it['injection']}"] for it in RS]).float().mean(0).half()   # mean clean head output per layer and position
def run_set(tag, items, target_prompt, source, conds, kinds=None):
    """Generate an answer per item for each named head set, cached per condition."""
    G = {c: jload(f'{OUT}/{U}/gen_{tag}_{c}.json') for c in conds}
    bl = chunks(by_len([target_prompt(it) for it in items]), BS)
    for bi, idx in enumerate(bl):
        bx = [items[i] for i in idx]; ps = [target_prompt(it) for it in bx]
        todo = [c for c in conds if any(it['key'] not in G[c] for it in bx)]
        for c in todo:
            kind = (kinds or {}).get(c, 'set')
            if kind == 'set': src = torch.stack([HA[source(it)] for it in bx])
            elif kind == 'mean': src = MEANC[None].expand(len(bx), -1, -1, -1); kind = 'set'
            else: src = None
            base = SETS[c.split('|')[0]]
            hk.spec = set_spec(base, src, kind='zero' if kind == 'zero' else 'set')
            try: outs, _ = generate_patched(m, ph, tok, ps, TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW)
            finally: hk.spec = {}
            G[c].update({it['key']: [o] for it, o in zip(bx, outs)}); jsave(G[c], f'{OUT}/{U}/gen_{tag}_{c}.json')
        if todo: print(f'  {tag}: batch {bi + 1}/{len(bl)} ({len(todo)} conditions)')
    return G
CTRLS = [f'random_{KMAIN}_draw{d + 1}' for d in range(RAND_DRAWS)] + [f'complement_{KMAIN}'] + ([f'attention_{KMAIN}'] if ATT_OK else [])
SUF_CONDS = [f'C_{k}' for k in KS] + CTRLS + ['all_heads']
GS = run_set('suff', TWIN_RS, lambda tw: tw['prompt'], lambda tw: BYINJ[tw['injection']]['key'], SUF_CONDS)
NEC_CONDS = list(SUF_CONDS)
GN = run_set('nec', RS, lambda it: it['prompt'], lambda it: f"twin{it['injection']}", NEC_CONDS)
GA = run_set('nec_abl', RS, lambda it: it['prompt'], lambda it: f"twin{it['injection']}", [f'C_{KMAIN}'], kinds={f'C_{KMAIN}': 'mean'})
GZ = run_set('nec_zero', RS, lambda it: it['prompt'], lambda it: f"twin{it['injection']}", [f'C_{KMAIN}'], kinds={f'C_{KMAIN}': 'zero'})
judge_pending([(BYINJ[tw['injection']]['category'], BYINJ[tw['injection']]['goal'], tw['document'], GS[c][tw['key']][0]) for c in GS for tw in TWIN_RS]
              + [(it['category'], it['goal'], it['document'], G[c][it['key']][0]) for G in (GN, GA, GZ) for c in G for it in RS])
free_seven_b()
def suff_rate(c):
    """Share of clean twins that follow the injection of their own injection's attacked run."""
    v = []
    for tw in TWIN_RS:
        src = BYINJ[tw['injection']]; v.append(ok(src['category'], src['goal'], tw['document'], GS[c][tw['key']][0]))
    return rate(v)
def nec_vec(G, c): return np.array([not ok(it['category'], it['goal'], it['document'], G[c][it['key']][0]) for it in RS], bool)
def content(G, c):
    st = nec_vec(G, c); outs = [G[c][it['key']][0] for it, s_ in zip(RS, st) if s_]; refs = [it['clean_answer'] for it, s_ in zip(RS, st) if s_]
    f1 = float(np.median([token_f1(o, r_) for o, r_ in zip(outs, refs)])) if outs else np.nan
    return f1, (float(np.mean([degenerate(o) for o in outs])) if outs else np.nan), len(outs)
srows = []
for c in SUF_CONDS:
    r_ = suff_rate(c); srows.append(dict(head_set=c, size=len(SETS[c]), twins_following_the_injection=rstr(r_), rate=round(r_['rate'], 3)))
SUFT = pd.DataFrame(srows); SUFT.to_csv(f'{REP}/sufficiency_sets.csv', index=False)
nrows = []
for c in NEC_CONDS:
    s_ = nec_vec(GN, c); f1, dg, k_ = content(GN, c)
    nrows.append(dict(head_set=c, size=len(SETS[c]), attacks_stopped=rstr(rate(s_)), rate=round(float(s_.mean()), 3), stopped_n=k_, median_f1_vs_clean_answer=round(f1, 3) if f1 == f1 else np.nan, degenerate=round(dg, 3) if dg == dg else np.nan))
for tagc, G in (('C_20 mean ablation', GA), ('C_20 zeroed', GZ)):
    s_ = nec_vec(G, f'C_{KMAIN}'); f1, dg, k_ = content(G, f'C_{KMAIN}')
    nrows.append(dict(head_set=tagc, size=KMAIN, attacks_stopped=rstr(rate(s_)), rate=round(float(s_.mean()), 3), stopped_n=k_, median_f1_vs_clean_answer=round(f1, 3) if f1 == f1 else np.nan, degenerate=round(dg, 3) if dg == dg else np.nan))
NECT = pd.DataFrame(nrows); NECT.to_csv(f'{REP}/necessity_sets.csv', index=False)
print('\n=== sufficiency: the attacked run\'s head outputs written into the clean twin (72 twins) ==='); print(SUFT.to_string(index=False))
print('\n=== necessity: the clean twin\'s head outputs written into the attacked run (72 items) ==='); print(NECT.to_string(index=False))
# gate two: the margin of the joint C_20 patch against the judge on these 72 sufficiency runs
fp = f'{OUT}/{U}/gate2.json'; G2C = jload(fp); todo = [tw for tw in TWIN_RS if tw['key'] not in G2C]
for bi, idx in enumerate(chunks(by_len([tw['prompt'] for tw in todo]), BS_SWEEP)):
    bx = [todo[i] for i in idx]
    ps = [p for tw in bx for p in (tw['prompt'], tw['prompt'])]
    aid = [v for tw in bx for v in (pad_ans(tok, BYINJ[tw['injection']]['greedy']), pad_ans(tok, tw['greedy']))]
    src = torch.stack([HA[BYINJ[tw['injection']]['key']] for tw in bx for _ in range(2)])
    lp = answer_logprob(m, hk, tok, ps, TASK_SYSTEM, SFX, aid, spec=set_spec(C_SET[KMAIN], src))
    for j, tw in enumerate(bx): G2C[tw['key']] = float(lp[2 * j] - lp[2 * j + 1])
    jsave(G2C, fp)
g2m = np.array([G2C[tw['key']] for tw in TWIN_RS]); g2y = np.array([ok(BYINJ[tw['injection']]['category'], BYINJ[tw['injection']]['goal'], tw['document'], GS[f'C_{KMAIN}'][tw['key']][0]) for tw in TWIN_RS], bool)
g2c = np.array([BYINJ[tw['injection']]['category'] for tw in TWIN_RS]); GATE2 = strat_auroc(g2m, g2y, g2c)
GATE2_OK = bool(GATE2 >= CRIT['gate2_auroc']) if 0 < g2y.sum() < len(g2y) else False
print(f'\ngate two: AUROC of the margin against the judge on the joint C_20 patch = {GATE2:.3f} (mark {CRIT["gate2_auroc"]:.2f}) -> {"pass" if GATE2_OK else "FAIL: the sweep ranking is not trustworthy at head size; the plan re-ranks the top 64 heads by generation in a separate notebook"}')
pd.DataFrame([dict(gate='two (after the sweep)', what='margin against the judge, joint C_20 patch into the clean twins', n=len(TWIN_RS), followed=int(g2y.sum()), auroc=round(GATE2, 3), mark=CRIT['gate2_auroc'], met=GATE2_OK)]
             ).to_csv(f'{REP}/proxy_gates_two.csv', index=False)

## Stage G. The instruction sweep and the overlap test

The same task is now given legitimately: the instruction sits in the user turn, the document is clean, the source run is that instruction run and the target is the plain summarisation run on the same document. Three disjoint document pools are drawn once: pool one for the instruction sweep, pool two for the I1 check and the cross-transfer, pool three is reserved for the defense notebook's utility set and is only written out here.

In [ ]:
INSTR = {}
for c, fam in FAMILIES.items():
    ins = sorted({SUB.instruction.iloc[i] for i in range(n) if SUB.meta.iloc[i] == c}); assert ins, c
    INSTR[fam] = dict(category=c, instruction=ins[0], n_distinct=len(ins))
print('instruction families (one BIPIA instruction each):')
for fam, d in INSTR.items(): print(f'  {fam} ({d["category"]}, {d["n_distinct"]} distinct in the set): {" ".join(d["instruction"].split())[:110]}')
docs = sorted(set(CLEAN_DOC)); r_ = np.random.default_rng(SEED + 1); perm = r_.permutation(len(docs))
POOL1 = [docs[i] for i in perm[:30]]; POOL2 = [docs[i] for i in perm[30:60]]; POOL3 = [docs[i] for i in perm[60:150]]
json.dump(dict(pool1=POOL1, pool2=POOL2, pool3=POOL3), open(f'{OUT}/{U}/document_pools.json', 'w'))
print('document pools:', len(POOL1), len(POOL2), len(POOL3), '(pool three is reserved for the defense notebook)')
def instr_items(pool, tag):
    """One item per (document, family): the instruction run, the plain run, and the answers both give unpatched."""
    its = []
    for di, d in enumerate(pool):
        for fam, info in INSTR.items():
            its.append(dict(key=f'{tag}:{di}:{fam}', family=fam, category=info['category'], goal=info['instruction'], document=d,
                            prompt=TASK_USER_WITH.format(instr=info['instruction'], doc=d), plain_prompt=TASK_USER.format(doc=d), plain_key=f'{tag}plain:{di}'))
    return its
I1_ITEMS, I2_ITEMS = instr_items(POOL1, 'p1'), instr_items(POOL2, 'p2')
fp = f'{OUT}/{U}/instr_base.json'; IB = jload(fp)
need = [it for it in I1_ITEMS + I2_ITEMS if it['key'] not in IB] + [it for it in I1_ITEMS + I2_ITEMS if it['plain_key'] not in IB]
if need:
    pr = {}
    for it in I1_ITEMS + I2_ITEMS: pr[it['key']] = it['prompt']; pr[it['plain_key']] = it['plain_prompt']
    ks = [k for k in pr if k not in IB]
    for idx in chunks(by_len([pr[k] for k in ks]), BS):
        outs, _ = generate_patched(m, ph, tok, [pr[ks[i]] for i in idx], TASK_SYSTEM, SFX, max_new_tokens=MAX_NEW)
        for j, i in enumerate(idx): IB[ks[i]] = outs[j]
        jsave(IB, fp)
    print('unpatched instruction and plain answers generated:', len(ks))
judge_pending([(it['category'], it['goal'], it['document'], IB[it['key']]) for it in I1_ITEMS + I2_ITEMS]
              + [(it['category'], it['goal'], it['document'], IB[it['plain_key']]) for it in I1_ITEMS + I2_ITEMS])
free_seven_b()
for it in I1_ITEMS + I2_ITEMS:
    it['greedy'] = IB[it['key']]; it['clean_answer'] = IB[it['plain_key']]
    it['instr_followed'] = ok(it['category'], it['goal'], it['document'], it['greedy']); it['plain_followed'] = ok(it['category'], it['goal'], it['document'], it['clean_answer'])
IFT = pd.DataFrame([dict(family=f, pool='one', n=sum(1 for it in I1_ITEMS if it['family'] == f), instruction_run_does_the_task=round(float(np.mean([it['instr_followed'] for it in I1_ITEMS if it['family'] == f])), 3),
                         plain_run_does_it=round(float(np.mean([it['plain_followed'] for it in I1_ITEMS if it['family'] == f])), 3)) for f in INSTR]
                   + [dict(family=f, pool='two', n=sum(1 for it in I2_ITEMS if it['family'] == f), instruction_run_does_the_task=round(float(np.mean([it['instr_followed'] for it in I2_ITEMS if it['family'] == f])), 3),
                           plain_run_does_it=round(float(np.mean([it['plain_followed'] for it in I2_ITEMS if it['family'] == f])), 3)) for f in INSTR])
IFT.to_csv(f'{REP}/instruction_baseline.csv', index=False); print('\n=== the legitimate instruction, unpatched (the sweep needs runs that do the task) ==='); print(IFT.to_string(index=False))
IOK = [it for it in I1_ITEMS if it['instr_followed'] and not it['plain_followed']]
print('pool one items where the instruction run does the task and the plain run does not:', len(IOK), 'of', len(I1_ITEMS))
assert len(IOK) >= 10, 'too few usable instruction items for a sweep; the instruction comparator cannot be built this way'
ISW = stratified([dict(it, category=it['family']) for it in IOK], min(N_INSTR_SWEEP, len(IOK)), SEED + 2)
ISW = [next(x for x in IOK if x['key'] == it['key']) for it in ISW]
print('instruction sweep items:', len(ISW), dict(collections.Counter(it['family'] for it in ISW)))
heads_of([(it['key'], it['prompt']) for it in I1_ITEMS + I2_ITEMS] + [(it['plain_key'], it['plain_prompt']) for it in I1_ITEMS + I2_ITEMS])
ISUF = sweep('instruction', ISW, lambda it: it['plain_prompt'], lambda it: it['key'], +1)
E_INS = eff(ISUF); I_SET = {k: [HEADS[i] for i in np.argsort(-E_INS)[:k]] for k in KS}
SW['instruction'] = E_INS; SW.to_csv(f'{REP}/head_effects.csv', index=False)
print('\n=== the 20 heads with the largest instruction effect ==='); print(SW.iloc[np.argsort(-E_INS)[:20]].round(4).to_string(index=False))
# I1 and the cross-transfer, on pool two
for k_, cset in (('I_20', I_SET[KMAIN]), ('C_20', C_SET[KMAIN])): SETS[k_ + '_instr'] = cset
I2OK = [it for it in I2_ITEMS if not it['plain_followed']]
GI = run_set('instr', I2OK, lambda it: it['plain_prompt'], lambda it: it['key'], ['I_20_instr', 'C_20_instr', 'all_heads'])
judge_pending([(it['category'], it['goal'], it['document'], GI[c][it['key']][0]) for c in GI for it in I2OK]); free_seven_b()
def instr_rate(c, items): return rate([ok(it['category'], it['goal'], it['document'], GI[c][it['key']][0]) for it in items])
I1R, CROSS, ALLR = instr_rate('I_20_instr', I2OK), instr_rate('C_20_instr', I2OK), instr_rate('all_heads', I2OK)
CRT = pd.DataFrame([dict(head_set='I_20 (instruction heads)', what='instruction run into the plain run, pool two', does_the_task=rstr(I1R), rate=round(I1R['rate'], 3)),
                    dict(head_set='C_20 (carrier heads)', what='instruction run into the plain run, pool two (cross-transfer)', does_the_task=rstr(CROSS), rate=round(CROSS['rate'], 3)),
                    dict(head_set='all heads', what='instruction run into the plain run, pool two', does_the_task=rstr(ALLR), rate=round(ALLR['rate'], 3))])
CRT.to_csv(f'{REP}/instruction_sets.csv', index=False); print('\n=== I1 and the cross-transfer (pool two, items the plain run does not already do) ==='); print(CRT.to_string(index=False))
# the overlap test
shared = sorted(set(C_SET[KMAIN]) & set(I_SET[KMAIN])); rho = float(pd.Series(E_SUF).rank().corr(pd.Series(E_INS).rank()))
r_ = np.random.default_rng(SEED + 3); lc = collections.Counter(l for l, _ in C_SET[KMAIN]); li = collections.Counter(l for l, _ in I_SET[KMAIN]); draws = []
for _ in range(10000):
    a = {(l, int(h)) for l, q in lc.items() for h in r_.permutation(hk.nh)[:q]}; b = {(l, int(h)) for l, q in li.items() for h in r_.permutation(hk.nh)[:q]}
    draws.append(len(a & b))
q999 = float(np.percentile(draws, 99.9))
O1 = ('shared' if (len(shared) >= CRIT['o1_shared_heads'] and len(shared) > q999 and rho >= CRIT['o1_rho'] and CROSS['rate'] >= CRIT['o1_cross'])
      else 'separate' if (len(shared) <= CRIT['o1_sep_heads'] and CROSS['rate'] <= CRIT['o1_sep_cross']) else 'partly shared')
OVR = [dict(scope='all families', shared_heads=len(shared), layer_matched_random_mean=round(float(np.mean(draws)), 2), layer_matched_99_9th=q999, spearman=round(rho, 3), cross_transfer=round(CROSS['rate'], 3), reading=O1)]
for f in INSTR:
    its = [it for it in I2OK if it['family'] == f]
    if its: OVR.append(dict(scope=f, shared_heads=len(shared), layer_matched_random_mean=np.nan, layer_matched_99_9th=np.nan, spearman=np.nan, cross_transfer=round(instr_rate('C_20_instr', its)['rate'], 3), reading=''))
OVT = pd.DataFrame(OVR); OVT.to_csv(f'{REP}/overlap.csv', index=False)
pd.DataFrame([dict(head_set=k, layer=l, head=h) for k in ('C_10', 'C_20', 'C_40', 'I_20_instr') for l, h in SETS[k]]).to_csv(f'{REP}/head_sets.csv', index=False)
print('\n=== O1: overlap of the carrier and instruction sets ==='); print(OVT.to_string(index=False))
print('shared heads:', shared)

## Stage H. The marks and what they decide

In [ ]:
import matplotlib.pyplot as plt
from reslog import log_result
I1_OK = bool(I1R['rate'] >= CRIT['i1_do'])
s20, s40 = suff_rate(f'C_{KMAIN}'), suff_rate('C_40')
ctrl_suff = {c: suff_rate(c)['rate'] for c in CTRLS}
ctrl_max = max(ctrl_suff.values())
L1_FULL = bool(s20['rate'] >= CRIT['l1_suff'] and ctrl_max <= CRIT['l1_ctrl'])
L1_PART = bool((not L1_FULL) and s40['rate'] >= CRIT['l1_suff_partial'] and ctrl_max <= CRIT['l1_ctrl'])
n20 = nec_vec(GN, f'C_{KMAIN}'); nrand = {d: nec_vec(GN, f'random_{KMAIN}_draw{d + 1}') for d in range(RAND_DRAWS)}
nrand_max = max(float(v.mean()) for v in nrand.values()); worst = max(range(RAND_DRAWS), key=lambda d: nrand[d].mean())
pairv = newcombe_paired(n20, nrand[worst])
L2_OK = bool(float(n20.mean()) >= CRIT['l2_stop'] and nrand_max <= CRIT['l2_ctrl'] and pairv[1] >= CRIT['l2_paired_lo'])
f1_20, dg_20, k20 = content(GN, f'C_{KMAIN}')
L3_OK = bool(f1_20 == f1_20 and f1_20 >= CRIT['l3_f1_ref'] - CRIT['l3_f1_tol'] and dg_20 <= CRIT['l3_degen'])
checks = [
    ('Proxy gate one', f'injection margin against the judge, version "{GATE_NAME}" (see change 6)', f'{PROXY:.3f} [{plo:.3f}, {phi:.3f}]', f'>= {CRIT["proxy_auroc"]:.2f}', PROXY_OK),
    ('Proxy gate one as pre-registered', 'the same against the judge on the cross-document transplant: failed, kept on the record', f'{a_c:.3f} [{lo_c:.3f}, {hi_c:.3f}]', f'>= {CRIT["proxy_auroc"]:.2f}', bool(a_c >= CRIT['proxy_auroc'])),
    ('Proxy gate two', 'margin against the judge, joint C_20 patch on the 72', f'{GATE2:.3f}', f'>= {CRIT["gate2_auroc"]:.2f}', GATE2_OK),
    ('L1', f'C_20 from the attacked run into the clean twin: twins following the injection (controls: {", ".join(CTRLS)})', f'{rstr(s20)} (controls, highest {ctrl_max:.3f}); C_40 {rstr(s40)}', '>= 0.30 with controls <= 0.05', L1_FULL),
    ('L1 (partial)', 'C_40 reaches 0.15 with controls at most 0.05', f'{s40["rate"]:.3f} (controls {ctrl_max:.3f})', '>= 0.15 with controls <= 0.05', L1_PART),
    ('L2', 'C_20 from the clean twin into the attacked run: attacks stopped; random matched; paired difference', f'{rstr(rate(n20))}; random {nrand_max:.3f}; {dstr(pairv)}', '>= 0.60, <= 0.15, lower end >= 0.25', L2_OK),
    ('L3', 'median token F1 of the stopped answers with the clean twin\'s answer; degenerate share', f'{f1_20:.3f} on {k20} answers (notebook 53 full state 0.597); degenerate {dg_20:.3f}', '>= 0.497 and <= 0.05', L3_OK),
    ('I1', 'I_20 from an instruction run into a plain run: the model does the task', rstr(I1R), '>= 0.30', I1_OK),
    ('O1', 'overlap reading (void if I1 fails)', O1 if I1_OK else 'void: I1 failed', 'reported', True)]
VT = pd.DataFrame([dict(mark=a, criterion=b, value=v, needed=m_, met=bool(ok_)) for a, b, v, m_, ok_ in checks]); VT.to_csv(f'{REP}/go_criteria.csv', index=False)
if not PROXY_OK: DECISION = 'STOP: the injection margin does not track the judge on the same-document full-state effect; the sweep ranking means nothing and the plan\'s generation fallback is needed'
elif not (L1_FULL or L1_PART): DECISION = 'CLOSE AT THE HEAD LEVEL: no set of 40 heads is sufficient, so the carrier is spread too widely for a head-level defense; the transplant stays an observation'
elif not L2_OK: DECISION = f'GO to notebook 57 on sufficiency alone (L1 {"full" if L1_FULL else "partial"}, L2 failed): the defense is judged on security only and no necessity claim is made'
else: DECISION = f'GO to notebook 57 (L1 {"full" if L1_FULL else "partial"}, L2 and L3 pass); overlap reading {O1 if I1_OK else "void, I1 failed: the mean-ablation form only"}'
GTXT = (f'{DECISION}\nproxy {"pass" if PROXY_OK else "fail"} / gate two {"pass" if GATE2_OK else "fail"}, L1 {"pass" if L1_FULL else "partial" if L1_PART else "fail"}, '
        f'L2 {"pass" if L2_OK else "fail"}, L3 {"pass" if L3_OK else "fail"}, I1 {"pass" if I1_OK else "fail"}, O1 {O1 if I1_OK else "void"}')
open(f'{REP}/go.txt', 'w').write(GTXT + '\n')
# by family, on the robust items
FAMT = pd.DataFrame([dict(family=c, robust_items=len(its), sufficiency_C20=round(float(np.mean([ok(BYINJ[tw['injection']]['category'], BYINJ[tw['injection']]['goal'], tw['document'], GS[f'C_{KMAIN}'][tw['key']][0]) for tw in its_tw])), 3) if its_tw else np.nan,
                          necessity_C20=round(float(np.mean([not ok(it['category'], it['goal'], it['document'], GN[f'C_{KMAIN}'][it['key']][0]) for it in its])), 3) if its else np.nan)
                     for c in CATEGORIES for its in [[it for it in RS if it['category'] == c]] for its_tw in [[tw for tw in TWIN_RS if BYINJ[tw['injection']]['category'] == c]]])
FAMT.to_csv(f'{REP}/by_family.csv', index=False)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
for j, (e, lab) in enumerate(((E_SUF, 'sufficiency'), (E_NEC, 'necessity'), (E_INS, 'instruction'))):
    M = e.reshape(len(hk.layers), hk.nh); im = ax[j].imshow(M, aspect='auto', cmap='RdBu_r', vmin=-np.abs(M).max(), vmax=np.abs(M).max())
    ax[j].set_title(f'{lab} effect on the injection margin', fontsize=10); ax[j].set_xlabel('head'); ax[j].set_ylabel('layer'); plt.colorbar(im, ax=ax[j], fraction=.046)
    for l, h in (C_SET[KMAIN] if lab != 'instruction' else I_SET[KMAIN]): ax[j].plot(h, l, 'k+', ms=5, mew=1)
plt.tight_layout(); plt.savefig(f'{FIG}/heads.png', dpi=150, bbox_inches='tight'); plt.show()
for title, T_ in (('MARKS', VT), ('SUFFICIENCY SETS', SUFT), ('NECESSITY SETS', NECT), ('INSTRUCTION AND CROSS-TRANSFER', CRT), ('OVERLAP', OVT), ('BY FAMILY', FAMT)):
    print(f'\n=== {title} ==='); print(T_.to_string(index=False))
print('\n' + GTXT)
summary = '\n\n'.join([GTXT, 'Marks:\n' + VT.to_string(index=False), 'Gate one, all three versions:\n' + GT1.to_string(index=False), 'Sufficiency:\n' + SUFT.to_string(index=False), 'Necessity:\n' + NECT.to_string(index=False),
                       'Instruction and cross-transfer:\n' + CRT.to_string(index=False), 'Overlap:\n' + OVT.to_string(index=False), 'By family:\n' + FAMT.to_string(index=False),
                       'Literature opened:\n' + LITT.to_string(index=False)])
log_result('nb56 (follow-on work): carrier heads, localisation and overlap on the 3B (exploration, nothing claimed)', summary, reports_dir=REP)

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", "nb56 (follow-on work): carrier heads, localisation and overlap on the 3B (exploration, nothing claimed). Per-head patching at the five template positions, on the input to each layer's attention output projection, in a new module src/head_patching.py (task_state.py and state_patching.py unchanged); items, screen and donors read from notebook 53's caches and checked against its committed per-item file; the injection margin validated against the judge on the full-state transplant before the sweep and on the joint C_20 patch after it; sufficiency and necessity sweeps over 576 heads on 32 robust items with set checks by generation and the paper's judge on all 72, against random layer-matched, lowest-effect-in-layer and attention-mass control sets, with mean ablation and zeroing recorded; the same sweep for a legitimate instruction in the user turn on three disjoint document pools, the I1 check, the cross-transfer and the overlap test against 10,000 layer-matched draws; marks and the decision rule fixed in the first cell; every paper of the plan's Table 1 opened in the first stage; reports/head_patching/*, figures/head_patching/heads.png"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)